# Foundation experiments

Один ноутбук для всех конфигураций. GPU T4 ×2, интернет выключен. Подключите данные соревнования и private dataset с `base.zip`.

Сначала выполните три контроля. Затем меняйте только SELECTED. Все 21 конфигурации за один 50-минутный запуск выполнять не нужно. Время каждой измеряется. Это исследовательский ноутбук для train/val, а не финальный train+val сабмит.

In [ ]:
SELECTED = ["019_base", "020_resnet", "029_siglip"]
SEEDS = None  # None uses each configuration seed; [421, 853] repeats selected runs.

In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile
import torch
assert torch.cuda.device_count() == 2, "Select GPU T4 x2"
archives = list(Path("/kaggle/input").rglob("base.zip"))
assert len(archives) == 1, "Attach exactly one private dataset containing base.zip"
project = Path("/kaggle/working/project")
project.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as bundle:
    for entry in bundle.infolist():
        dest = (project / entry.filename).resolve()
        assert dest.is_relative_to(project.resolve()), "Unsafe archive path"
    bundle.extractall(project)
manifest = json.loads((project / "package.json").read_text())
for name, digest in manifest["files"].items():
    with (project / name).open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == digest, name
sys.path.insert(0, str(project))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
roots = [p.parent for p in Path("/kaggle/input").rglob("train.csv")
         if (p.parent / "val.csv").is_file() and (p.parent / "train").is_dir() and (p.parent / "val").is_dir()]
assert len(roots) == 1, "Competition CSVs and train/val image directories must share one root"
data = roots[0]
print("Project:", project, "Data:", data)


Все модели, цели и PCA обучаются только на train. В каждой новой конфигурации вторая ветка смеси фиксируется на историческом контроле. Результаты и веса записываются в `artefacts`, временные файлы — в `/kaggle/temp/foundation`.

In [ ]:
from src.suite import run_suite
results = run_suite(project=project, data=data, inputs=project / "inputs",
                    selected=SELECTED, seeds=SEEDS,
                    work=Path("/kaggle/temp/foundation"),
                    output=Path("/kaggle/working/artefacts"))
results
